In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

import networkx as nx

In [2]:
# ---------- Goal of this script ----------
# The goal of this script is to build weighted, directed graph

In [3]:
# ---------- Paths ----------

BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "Data"

DATA_FILE = DATA_DIR / "final_trip_data.parquet"

In [4]:
trip_data = pd.read_parquet(DATA_FILE)
trip_data.head()

,trip_id,start_date,start_station_id,start_station_name,end_date,end_station_id,end_station_name,bike_id,bike_model,total_duration,total_duration_ms,start_lat,start_lon,end_lat,end_lon
0,136666627,2024-01-14 23:59:00,1108,"North Wharf Road, Paddington",2024-01-15 00:06:00,3423.0,"Maida Vale, Maida Vale",53020.0,CLASSIC,6m 47s,407799.0,51.518623,-0.17660300000000007,51.529857,-0.18348604
1,136666625,2024-01-14 23:57:00,3447,"Gloucester Road (North), Kensington",2024-01-15 00:05:00,1214.0,"Kensington Olympia Station, Olympia",54559.0,CLASSIC,8m 1s,481276.0,51.49792478,-0.183834706,51.49815779,-0.209494128
2,136666626,2024-01-14 23:57:00,1090,"Warren Street Station, Euston",2024-01-15 00:02:00,200005.0,"New Cavendish Street, Marylebone",52133.0,CLASSIC,5m 11s,311788.0,51.52443845,-0.138019439,51.519167,-0.147983
3,136666622,2024-01-14 23:56:00,200058,"Northdown Street, King's Cross",2024-01-15 00:06:00,2687.0,"Brick Lane Market, Shoreditch",60341.0,PBSC_EBIKE,10m 14s,614030.0,51.531066,-0.11934,51.52261762,-0.071653961
4,136666623,2024-01-14 23:56:00,1052,"Soho Square , Soho",2024-01-15 00:23:00,1150.0,"Queensway, Kensington Gardens",55212.0,CLASSIC,27m 1s,1621583.0,51.51563144,-0.132328837,51.51031,-0.18740235


In [5]:
# ---------- Stations - nodes dataframe ----------
# Każdy wierzchołek ma:
# - Numer stacji
# - Nazwa stacji 
# - Położenie geograficzne (dwie współrzędne)

start_stations_df = trip_data[
    [
        "start_station_id",
        "start_station_name",
        "start_lat",
        "start_lon",
    ]
].drop_duplicates()

start_stations_df = start_stations_df.rename(columns={
    "start_station_id": "station_id",
    "start_station_name": "station_name",
    "start_lat": "lat",
    "start_lon": "lon"
})

start_stations_df.head()

,station_id,station_name,lat,lon
0,1108,"North Wharf Road, Paddington",51.518623,-0.17660300000000007
1,3447,"Gloucester Road (North), Kensington",51.49792478,-0.183834706
2,1090,"Warren Street Station, Euston",51.52443845,-0.138019439
3,200058,"Northdown Street, King's Cross",51.531066,-0.11934
4,1052,"Soho Square , Soho",51.51563144,-0.132328837


In [6]:
end_stations_df = trip_data[
    [
        "end_station_id",
        "end_station_name",
        "end_lat",
        "end_lon",
    ]
].drop_duplicates()

end_stations_df = end_stations_df.rename(columns={
    "end_station_id": "station_id",
    "end_station_name": "station_name",
    "end_lat": "lat",
    "end_lon": "lon"
})

end_stations_df.head()

,station_id,station_name,lat,lon
0,3423.0,"Maida Vale, Maida Vale",51.529857,-0.18348604
1,1214.0,"Kensington Olympia Station, Olympia",51.49815779,-0.209494128
2,200005.0,"New Cavendish Street, Marylebone",51.519167,-0.147983
3,2687.0,"Brick Lane Market, Shoreditch",51.52261762,-0.071653961
4,1150.0,"Queensway, Kensington Gardens",51.51031,-0.18740235


In [7]:
stations_df = pd.concat([start_stations_df, end_stations_df], ignore_index=True).drop_duplicates()
print("Number of stations:", len(stations_df))

Number of stations: 801


In [8]:
# Krawędzie A --> B mają wagę równą liczbie przejazdów z A do B
edges_df = trip_data.groupby(
    [
        "start_station_id",
        "start_station_name",
        "end_station_id",
        "end_station_name"
    ]
).agg(
    weight=("start_station_id", "count")
).reset_index()

In [9]:
edges_df.head(-10)

,start_station_id,start_station_name,end_station_id,end_station_name,weight
0,959,"Milroy Walk, South Bank",959.0,"Milroy Walk, South Bank",296
1,959,"Milroy Walk, South Bank",960.0,"Hop Exchange, The Borough",622
2,959,"Milroy Walk, South Bank",961.0,"Union Street, The Borough",159
3,959,"Milroy Walk, South Bank",962.0,"Stamford Street, South Bank",221
4,959,"Milroy Walk, South Bank",963.0,"Bankside Mix, Bankside",231
...,...,...,...,...,...
499733,300253,"Bermondsey Station, Bermondsey",300236.0,"Bevington Road West, North Kensington",1
499734,300253,"Bermondsey Station, Bermondsey",300237.0,"Tate Modern, Bankside",42
499735,300253,"Bermondsey Station, Bermondsey",300238.0,"Southwark Street, Bankside",8
499736,300253,"Bermondsey Station, Bermondsey",300239.0,"Victory Place, Walworth",22


In [10]:
graph = nx.from_pandas_edgelist(
    edges_df,
    source="start_station_name",
    target="end_station_name",
    edge_attr="weight",
    create_using=nx.DiGraph()
)

positions = {}

stations_df["lat"] = stations_df["lat"].astype(float)
stations_df["lon"] = stations_df["lon"].astype(float)

for _, row in stations_df.iterrows():

    positions[row["station_name"]] = (
        row["lon"],
        row["lat"]
    )

In [11]:
graph.in_degree()

InDegreeView({'Milroy Walk, South Bank': 703, 'Hop Exchange, The Borough': 791, 'Union Street, The Borough': 693, 'Stamford Street, South Bank': 699, 'Bankside Mix, Bankside': 755, "Bath Street, St. Luke's": 729, 'Tachbrook Street, Victoria': 700, 'New Kent Road, The Borough': 685, 'Golden Lane, Barbican': 671, 'Warwick Row, Westminster': 771, 'Gower Place , Euston': 720, 'Scala Street, Fitzrovia': 682, "Godliman Street, St. Paul's": 757, 'Hampstead Road, Euston': 725, 'Bethnal Green Road, Shoreditch': 730, 'Guilford Street , Bloomsbury': 706, "Theobald's Road , Holborn": 716, "Longford Street, The Regent's Park": 599, 'Russell Square Station, Bloomsbury': 740, 'Wenlock Road , Hoxton': 608, 'Malet Street, Bloomsbury': 760, 'British Museum, Bloomsbury': 748, 'Holborn Circus, Holborn': 777, 'Euston Road, Euston': 687, 'Finsbury Circus, Liverpool Street': 768, 'Hatton Garden, Holborn': 714, 'Finsbury Library , Finsbury': 609, 'Great Russell Street, Bloomsbury': 749, 'Murray Grove , Hoxton

In [12]:
nx.is_weakly_connected(graph)

True

In [13]:
nx.is_strongly_connected(graph)

True

In [14]:
np.mean([degree for _, degree in graph.degree()])

np.float64(1247.8102372034957)

In [15]:
nx.pagerank(graph, weight="weight")

{'Milroy Walk, South Bank': 0.0011678072314575587,
 'Hop Exchange, The Borough': 0.004480859839425038,
 'Union Street, The Borough': 0.001170377910722598,
 'Stamford Street, South Bank': 0.0012799473407231865,
 'Bankside Mix, Bankside': 0.0015595731867833339,
 "Bath Street, St. Luke's": 0.0016270011093740661,
 'Tachbrook Street, Victoria': 0.0013035342191816555,
 'New Kent Road, The Borough': 0.0012039462213224205,
 'Golden Lane, Barbican': 0.0011454908803765968,
 'Warwick Row, Westminster': 0.002232261148623905,
 'Gower Place , Euston': 0.0012078433364366534,
 'Scala Street, Fitzrovia': 0.001037743132402183,
 "Godliman Street, St. Paul's": 0.0015192221259802809,
 'Hampstead Road, Euston': 0.0017204003087401476,
 'Bethnal Green Road, Shoreditch': 0.0025663031695116654,
 'Guilford Street , Bloomsbury': 0.001689110218386588,
 "Theobald's Road , Holborn": 0.0012979473792098936,
 "Longford Street, The Regent's Park": 0.000819532280368342,
 'Russell Square Station, Bloomsbury': 0.0015129534